In [1]:
import os, shutil, subprocess, time, sys

# 1. Setup Folders and cleanup
PROJECT_DIR = '/content/LLM_KG'
if os.path.exists(PROJECT_DIR): shutil.rmtree(PROJECT_DIR)

# 2. Clone YOUR Fork & Install dependencies
# Replace with your fork link if different
!git clone https://github.com/KartavyaDikshit/LLM_KG.git
%cd {PROJECT_DIR}

!pip install -r requirements.txt --quiet
!pip install langchain-community langchain-ollama pyvis tabulate PyYAML --quiet
sys.path.append(PROJECT_DIR)

# 3. Setup Ollama (Local Engine)
!sudo apt-get update && sudo apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh
subprocess.Popen(["ollama", "serve"])
time.sleep(15)

# 4. Pull ONLY the most stable models
models = ["llama3", "mistral", "gemma2"]
for m in models:
    print(f"📥 Pulling {m}...")
    !ollama pull {m}

print("✅ Environment Ready!")

Cloning into 'LLM_KG'...
remote: Enumerating objects: 187, done.
remote: Counting objects: 100% (187/187), done.
remote: Compressing objects: 100% (134/134), done.
remote: Total 187 (delta 93), reused 141 (delta 47), pack-reused 0 (from 0)
Receiving objects: 100% (187/187), 378.28 KiB | 10.51 MiB/s, done.
Resolving deltas: 100% (93/93), done.
/content/LLM_KG
Hit:1 http://security.ubuntu.com/ubuntu jammy-security InRelease
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:6 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Hit:9 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Fetched 3,917 B in 1s (6,304 B/s)
Readi

In [2]:
import os

# Create Domain Config Directory
os.makedirs('src/config/domains', exist_ok=True)

# 1. Write Medical Config
with open('src/config/domains/medical.yaml', 'w') as f:
    f.write("""
domain_name: "medical"
description: "Clinical notes and medical relationships"
entity_types: ["Drug", "Disease", "Symptom", "Procedure"]
allowed_predicates: ["TREATS", "CAUSES", "DIAGNOSES", "HAS_SYMPTOM"]
planner_instruction: "Identify medications, dosages, conditions, and symptoms."
extractor_instruction: "Extract a high-density medical knowledge graph."
color_map:
  drug: "#3b82f6"
  disease: "#ef4444"
  symptom: "#f59e0b"
  patient: "#6b7280"
""")

# 2. Write Legal Config
with open('src/config/domains/legal.yaml', 'w') as f:
    f.write("""
domain_name: "legal"
description: "Commercial contracts and legal obligations"
entity_types: ["Party", "Agreement", "Obligation", "Jurisdiction"]
allowed_predicates: ["BOUND_BY", "GOVERNED_BY", "MANDATES"]
planner_instruction: "Identify parties, their obligations, and governing laws."
extractor_instruction: "Extract a legal knowledge graph focusing on who owes what."
color_map:
  party: "#1e293b"
  agreement: "#0f172a"
  obligation: "#dc2626"
  jurisdiction: "#059669"
""")

# 3. Apply Multi-Domain Node Logic
patch_code = r'''
import os, json, re, yaml
try:
    from langchain_ollama import ChatOllama
except ImportError:
    from langchain_community.chat_models import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from src.agents.state import AgentState, Triple

def load_domain_config(domain):
    path = f"src/config/domains/{domain}.yaml"
    with open(path, "r") as f: return yaml.safe_load(f)

def get_llm(model_name="llama3"):
    return ChatOllama(model=model_name, temperature=0)

def planner_node(state, config=None):
    cfg = load_domain_config(state.get("domain", "medical"))
    llm = config.get("configurable", {}).get("llm", get_llm()) if config else get_llm()
    prompt = ChatPromptTemplate.from_template("Domain: {name}\nGoal: {instr}\nText: {text}")
    res = (prompt | llm).invoke({"name": cfg["domain_name"], "instr": cfg["planner_instruction"], "text": state["input_text"]})
    return {"planner_strategy": res.content, "iterations": state.get("iterations", 0) + 1}

def extractor_node(state, config=None):
    cfg = load_domain_config(state.get("domain", "medical"))
    llm = config.get("configurable", {}).get("llm", get_llm()) if config else get_llm()
    prompt = ChatPromptTemplate.from_template("Extract JSON triples for {name}.\nStrategy: {strat}\nText: {text}\nFormat: {{'triples': [{{'subject': '', 'predicate': '', 'obj': '', 'confidence': 1.0}}]}}")
    res = (prompt | llm).invoke({"name": cfg["domain_name"], "strat": state["planner_strategy"], "text": state["input_text"]})
    triples = []
    match = re.search(r"(\{.*\}|\[.*\])", res.content, re.DOTALL)
    if match:
        try:
            data = json.loads(match.group(0))
            items = data.get("triples", []) if isinstance(data, dict) else data
            for i in items: triples.append(Triple(subject=str(i.get("subject", "Unknown")), predicate=str(i.get("predicate", "RELATED_TO")), obj=str(i.get("obj", "Unknown")), confidence=1.0))
        except: pass
    return {"extracted_triples": triples}

def validator_node(state, config=None):
    return {"is_valid": True}
'''

with open('src/agents/nodes.py', 'w') as f: f.write(patch_code)

# 4. Update Visualizer for Multi-Domain
vis_patch = r'''
from pyvis.network import Network
import os, yaml

def visualize_graph(nx_graph, domain="medical", output_path="data/processed/kg.html"):
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    net = Network(height="550px", width="100%", bgcolor="#ffffff", directed=True)
    cfg_path = f"src/config/domains/{domain}.yaml"
    color_map = {}
    if os.path.exists(cfg_path):
        with open(cfg_path, "r") as f: color_map = yaml.safe_load(f).get("color_map", {})

    for node in nx_graph.nodes():
        color = "#94a3b8"
        for k, v in color_map.items():
            if k in str(node).lower(): color = v; break
        net.add_node(node, label=str(node), color=color)
    for s, t, d in nx_graph.edges(data=True):
        net.add_edge(s, t, label=d.get("label", ""))
    net.save_graph(output_path)
'''
with open('src/graph/visualizer.py', 'w') as f: f.write(vis_patch)

print("✅ Multi-Domain Implementation Patched!")

✅ Multi-Domain Implementation Patched!


In [3]:
import os, base64
from src.agents.graph import create_agentic_workflow
from src.agents.nodes import get_llm
from src.graph.builder import GraphBuilder
from src.graph.visualizer import visualize_graph
from IPython.display import HTML, display

def render_colab_kg(path, title):
    if not os.path.exists(path): return
    with open(path, 'r', encoding='utf-8') as f: html = f.read()
    b64 = base64.b64encode(html.encode('utf-8')).decode('utf-8')
    iframe = f'<div style="border:1px solid #ddd; border-radius:8px; padding:10px; margin-bottom:20px;"><h3>🔍 {title}</h3><iframe src="data:text/html;base64,{b64}" width="100%" height="600px" style="border:none;"></iframe></div>'
    display(HTML(iframe))

workflow = create_agentic_workflow()
llm = get_llm("llama3")

# --- TEST 1: MEDICAL ---
print("🔬 Running Medical Extraction...")
med_text = "Patient with acute myocardial infarction prescribed Aspirin and Lisinopril."
res_med = workflow.invoke({"input_text": med_text, "domain": "medical", "is_valid": False, "iterations": 0}, config={"configurable": {"llm": llm}})
builder_med = GraphBuilder()
builder_med.add_triples(res_med["extracted_triples"])
visualize_graph(builder_med.graph, domain="medical", output_path="data/processed/med_kg.html")
render_colab_kg("data/processed/med_kg.html", "Medical KG (Llama 3)")

# --- TEST 2: LEGAL ---
print("⚖️ Running Legal Extraction...")
legal_text = "ACME Corp is bound by the laws of Delaware. This Agreement mandates payment to Beta Inc by June 1st."
res_leg = workflow.invoke({"input_text": legal_text, "domain": "legal", "is_valid": False, "iterations": 0}, config={"configurable": {"llm": llm}})
builder_leg = GraphBuilder()
builder_leg.add_triples(res_leg["extracted_triples"])
visualize_graph(builder_leg.graph, domain="legal", output_path="data/processed/legal_kg.html")
render_colab_kg("data/processed/legal_kg.html", "Legal KG (Llama 3)")


🔬 Running Medical Extraction...


⚖️ Running Legal Extraction...
